# Demo vs no demo: does showing the task once help?

**One-line claim (to be checked against the data below):** with the same generator, planner and repair budget, an arm that sees one
demonstration (plus a scene description and contact evidence) produces programs that transfer to an unseen object scale,
while a task-text-only arm does not. All numbers in this notebook are computed from the logged `events.jsonl` files of the two runs.

**Design.** Same task (`put the bowl on the plate`, `libero_goal_swap` task 8), same model (`muse-wonder-1.0`), same planner
(`curobo-sim`: cuRobo with the robot mask from the simulator, held bowl attached with its known size), up to 4 repairs, debugging on
seed 51 in the scaled scene (bowl x1.227, plate x1.855). Two samples per arm (temperature 0.2 and 0.7). Evaluation of each final
program on held-out seeds 1 to 5 in the scaled scene (where it was debugged) and in the stock scene (never seen during repair).

| | Demo arm | No-demo arm |
|---|---|---|
| Demonstration (DemoSpec, clip 233859) | yes | no |
| Scene description S4 in every prompt | yes | no |
| Contact evidence E3 in every repair | yes | no |

**The arms differ in three things, not one.** The comparison measures "demo + scene + evidence" against "task text only". Separating the
demonstration from the other two needs a third arm (demo, no scene, no evidence); see section 6.

In [1]:
import json, math, random
from collections import defaultdict
from pathlib import Path

HERE = Path.cwd()
ROOT = next((p for p in (HERE, HERE / "docs" / "synthesizer" / "show-once-vs-task-only") if (p / "demo" / "events.jsonl").exists()), None)
assert ROOT is not None, "open this notebook from docs/synthesizer/show-once-vs-task-only or from the repo root"

def load_events(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

ARMS = ("demo", "nodemo")
EV = {a: load_events(ROOT / a / "events.jsonl") for a in ARMS}
CFG = {a: json.loads((ROOT / a / "config.json").read_text(encoding="utf-8")) for a in ARMS}
for a in ARMS:
    c = CFG[a]
    print(f"{a:7s} planner={c['planner']} attach_known={c['attach_known']} evidence={c['evidence']} scene_text={c['scene_text_file']} "
          f"repairs={c['repairs']} temps={c['temperatures']} eval_seeds={c['eval_seeds']} budget={c['token_budget']}")

demo    planner=curobo-sim attach_known=True evidence=e3 scene_text=data/scene_text/S4_L1.txt repairs=4 temps=0.2,0.7 eval_seeds=[1, 2, 3, 4, 5] budget=300000
nodemo  planner=curobo-sim attach_known=True evidence=none scene_text=None repairs=4 temps=0.2,0.7 eval_seeds=[1, 2, 3, 4, 5] budget=300000


In [2]:
# tidy records
evals = [dict(arm=e["arm"], sample=e["sample"], scene=e["scene"], seed=e["seed"], ok=bool(e["success"]), reason=e["reason"])
         for a in ARMS for e in EV[a] if e["kind"] == "eval"]
temps = {(a, e["sample"]): e["temperature"] for a in ARMS for e in EV[a] if e["kind"] == "sample_start"}
rollouts = [e for a in ARMS for e in EV[a] if e["kind"] == "rollout"]
programs = [e for a in ARMS for e in EV[a] if e["kind"] == "program"]
run_end = {a: next(e for e in EV[a] if e["kind"] == "run_end") for a in ARMS}
SCENES = sorted({r["scene"] for r in evals})
print("scenes:", SCENES, "| eval rows:", len(evals), "| rollouts:", len(rollouts), "| programs:", len(programs))

scenes: ['scaled', 'stock'] | eval rows: 40 | rollouts: 12 | programs: 12


In [3]:
# --- small exact statistics, standard library only ---
def _bisect(f, lo=0.0, hi=1.0, it=200):
    flo = f(lo)
    for _ in range(it):
        mid = (lo + hi) / 2
        if (f(mid) > 0) == (flo > 0):
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

def binom_cdf(k, n, p):
    return sum(math.comb(n, i) * p**i * (1 - p) ** (n - i) for i in range(k + 1))

def wilson(k, n, z=1.959964):
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def clopper_pearson(k, n, alpha=0.05):
    lo = 0.0 if k == 0 else _bisect(lambda p: (1 - binom_cdf(k - 1, n, p)) - alpha / 2)
    hi = 1.0 if k == n else _bisect(lambda p: binom_cdf(k, n, p) - alpha / 2)
    return (lo, hi)

def fisher_exact(a, b, c, d):
    # two-sided exact test for [[a, b], [c, d]]: a/b successes/failures of arm 1, c/d of arm 2
    n1, n2, K = a + b, c + d, a + c
    N = n1 + n2
    def pmf(x):
        return math.comb(K, x) * math.comb(N - K, n1 - x) / math.comb(N, n1)
    p_obs = pmf(a)
    return min(1.0, sum(pmf(x) for x in range(max(0, n1 - (N - K)), min(K, n1) + 1) if pmf(x) <= p_obs * (1 + 1e-9)))

def newcombe_diff(k1, n1, k2, n2):
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = wilson(k1, n1)
    l2, u2 = wilson(k2, n2)
    d = p1 - p2
    return d, d - math.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), d + math.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)

def pct(x):
    return f"{100 * x:5.1f}%"

## At a glance

Hover any mark for details. The figures are also saved in `figures/` for slides.

> **Before quoting these numbers:** the two arms differ in more than the demonstration (scene text and contact evidence too), and both ran before the
> attach frame fix (section 6). Treat this as a signal from a small sample.

In [4]:
# Chart helpers: plain SVG, no plotting dependency. Hover any mark for details. Each figure is also saved to figures/.
from html import escape
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None

DEMO, NODEMO = "#2a78d6", "#eb6834"
ARM_COLOR = {"demo": DEMO, "nodemo": NODEMO}
ARM_NAME = {"demo": "Demo", "nodemo": "No demo"}
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
PASS, FAIL = "#0c7a54", "#c23a39"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

def text(x, y, s, size=13, color=INK2, anchor="start", weight=400):
    return f'<text x="{x:.1f}" y="{y:.1f}" font-size="{size}" fill="{color}" text-anchor="{anchor}" font-weight="{weight}">{escape(str(s))}</text>'

def line(x1, y1, x2, y2, color=GRID, width=1, dash=""):
    d = f' stroke-dasharray="{dash}"' if dash else ""
    return f'<line x1="{x1:.1f}" y1="{y1:.1f}" x2="{x2:.1f}" y2="{y2:.1f}" stroke="{color}" stroke-width="{width}" stroke-linecap="round"{d}/>'

def tip(mark, s):
    return f"<g><title>{escape(s)}</title>{mark}</g>"

def legend(x, y, items):
    out = []
    for color, name in items:
        out.append(f'<circle cx="{x + 6}" cy="{y - 4}" r="6" fill="{color}"/>' + text(x + 18, y, name, 13, INK))
        x += 34 + 8 * len(name)
    return "".join(out)

def figure(name, title, subtitle, body, w=760, h=300):
    head = text(20, 30, title, 16, INK, weight=600) + "".join(text(20, 52 + 18 * i, s, 13) for i, s in enumerate(subtitle))
    svg = (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {w} {h}" width="100%" style="max-width:{w}px" '
           f'font-family="system-ui,-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif" role="img" aria-label="{escape(title)}">'
           f'<rect width="{w}" height="{h}" rx="8" fill="{SURFACE}"/>{head}{body}</svg>')
    (FIG_DIR / f"{name}.svg").write_text(svg, encoding="utf-8")
    if display:
        display(HTML(svg))
    else:
        print("saved", FIG_DIR / f"{name}.svg")

def tally(scene, arm):
    rs = [r["ok"] for r in evals if r["arm"] == arm and r["scene"] == scene]
    return sum(rs), len(rs)

SCENE_NAME = {"scaled": "Scaled scene: real-size objects, where programs were repaired", "stock": "Stock scene: other object sizes, never seen in repair"}

In [5]:
# Figure 1: the headline
def fig_rates():
    x0, x1, top = 110, 650, 112
    def X(p):
        return x0 + (x1 - x0) * p
    b = legend(20, 92, [(DEMO, "Demo + scene text + contact evidence"), (NODEMO, "No demo: task text only")])
    y = top
    for scene in SCENES:
        b += text(20, y + 8, SCENE_NAME[scene], 13, INK, weight=600)
        y += 30
        for a in ARMS:
            k, n = tally(scene, a)
            lo, hi = clopper_pearson(k, n)
            b += text(x0 - 12, y + 5, ARM_NAME[a], 13, INK, "end")
            b += line(X(lo), y, X(hi), y, ARM_COLOR[a], 2)
            b += tip(f'<circle cx="{X(k / n):.1f}" cy="{y}" r="8" fill="{ARM_COLOR[a]}" stroke="{SURFACE}" stroke-width="2"/>',
                     f"{ARM_NAME[a]}, {scene}: {k} of {n} passed. Exact 95% interval {100 * lo:.0f}% to {100 * hi:.0f}%.")
            b += text(X(hi) + 12, y + 5, f"{k}/{n}", 14, INK, weight=600)
            y += 30
        y += 8
    for p in (0, .25, .5, .75, 1):
        b = line(X(p), top + 18, X(p), y - 8) + b + text(X(p), y + 8, f"{p:.0%}", 12, INK2, "middle")
    b += text((x0 + x1) / 2, y + 28, "Share of held-out runs that passed (line: exact 95% interval)", 12, INK2, "middle")
    (kd, nd), (kn, nn) = tally("stock", "demo"), tally("stock", "nodemo")
    (sd, sn), (rd, rn) = tally("scaled", "demo"), tally("scaled", "nodemo")
    figure("fig1_success_rates", f"Stock scene: demo {kd}/{nd}, no demo {kn}/{nn}. Scaled scene: demo {sd}/{sn}, no demo {rd}/{rn}.",
           ["Final programs on held-out seeds 1 to 5. Small sample: read the intervals, not just the dots."], b, h=y + 46)
fig_rates()

In [6]:
# Figure 2: every single evaluation
def fig_grid():
    cw, ch, gap, left, top = 40, 32, 4, 160, 118
    b, x = "", left
    samples = [(a, s) for a in ARMS for s in sorted({r["sample"] for r in evals if r["arm"] == a})]
    for scene in SCENES:
        seeds = sorted({r["seed"] for r in evals if r["scene"] == scene})
        b += text(x, top - 30, f"{scene.capitalize()} scene", 13, INK, weight=600)
        for j, seed in enumerate(seeds):
            b += text(x + j * (cw + gap) + cw / 2, top - 8, f"seed {seed}" if j == 0 else seed, 12, INK2, "middle")
        for i, (a, s) in enumerate(samples):
            y = top + i * (ch + gap) + (10 if a == "nodemo" else 0)
            if x == left:
                b += f'<circle cx="28" cy="{y + ch / 2}" r="6" fill="{ARM_COLOR[a]}"/>' + text(40, y + ch / 2 + 5, f"{ARM_NAME[a]}, T={temps[(a, s)]}", 13, INK)
            hits = 0
            for j, seed in enumerate(seeds):
                r = next(r for r in evals if (r["arm"], r["sample"], r["scene"], r["seed"]) == (a, s, scene, seed))
                hits += r["ok"]
                cx = x + j * (cw + gap)
                mark = (f'<rect x="{cx}" y="{y}" width="{cw}" height="{ch}" rx="4" fill="{PASS if r["ok"] else "#f6e3e1"}"/>'
                        + text(cx + cw / 2, y + ch / 2 + 5, "✓" if r["ok"] else "✕", 15, "#ffffff" if r["ok"] else FAIL, "middle", 700))
                b += tip(mark, f'{ARM_NAME[a]}, T={temps[(a, s)]}, {scene}, seed {seed}: {"passed" if r["ok"] else "failed"} ({r["reason"]})')
            b += text(x + len(seeds) * (cw + gap) + 2, y + ch / 2 + 5, f"{hits}/{len(seeds)}", 13, INK, weight=600)
        x += len(seeds) * (cw + gap) + 44
    figure("fig2_every_run", "Every held-out run, one square each", ["✓ the bowl ended on the plate, ✕ it did not. Same five seeds for every program."],
           b, w=x - 10, h=top + len(samples) * (ch + gap) + 28)
fig_grid()

## 1. Success rates on held-out seeds

Pooled over the two samples and the five seeds (10 evaluations per cell). The 95% intervals are exact (Clopper-Pearson) and Wilson.
**Caution:** the 10 evaluations are not independent (the same 5 seeds are used by both samples, and each sample's program is one draw), so
these intervals are optimistic. Section 2 repeats the comparison with seeds as the unit.

In [7]:
rows = []
for scene in SCENES:
    for a in ARMS:
        k = sum(r["ok"] for r in evals if r["arm"] == a and r["scene"] == scene)
        n = sum(1 for r in evals if r["arm"] == a and r["scene"] == scene)
        cp, w = clopper_pearson(k, n), wilson(k, n)
        rows.append((scene, a, k, n, k / n, cp, w))
print(f"{'scene':8s} {'arm':7s} {'k/n':>6s} {'rate':>7s}  {'Clopper-Pearson 95%':>22s}  {'Wilson 95%':>16s}")
for scene, a, k, n, p, cp, w in rows:
    print(f"{scene:8s} {a:7s} {k:>3d}/{n:<2d} {pct(p)}  [{pct(cp[0])}, {pct(cp[1])}]  [{pct(w[0])}, {pct(w[1])}]")

print()
print("per sample (rate on 5 held-out seeds):")
for scene in SCENES:
    for a in ARMS:
        cells_ = []
        for s in sorted({r["sample"] for r in evals if r["arm"] == a}):
            rs = [r["ok"] for r in evals if r["arm"] == a and r["scene"] == scene and r["sample"] == s]
            cells_.append(f"T={temps[(a, s)]}: {sum(rs)}/{len(rs)}")
        print(f"  {scene:8s} {a:7s} " + " | ".join(cells_))

scene    arm        k/n    rate     Clopper-Pearson 95%        Wilson 95%
scaled   demo      7/10  70.0%  [ 34.8%,  93.3%]  [ 39.7%,  89.2%]
scaled   nodemo    8/10  80.0%  [ 44.4%,  97.5%]  [ 49.0%,  94.3%]
stock    demo      7/10  70.0%  [ 34.8%,  93.3%]  [ 39.7%,  89.2%]
stock    nodemo    0/10   0.0%  [  0.0%,  30.8%]  [  0.0%,  27.8%]

per sample (rate on 5 held-out seeds):
  scaled   demo    T=0.2: 5/5 | T=0.7: 2/5
  scaled   nodemo  T=0.2: 4/5 | T=0.7: 4/5
  stock    demo    T=0.2: 4/5 | T=0.7: 3/5
  stock    nodemo  T=0.2: 0/5 | T=0.7: 0/5


## 2. Arm comparison

Two views: (a) a pooled 2x2 table with Fisher's exact test and a Newcombe interval for the difference, which treats the 10 evaluations
per arm as independent; (b) a **paired analysis with seeds as the unit**, which is the safer one because both arms are evaluated on the same
seeds. For (b), each seed gets a score per arm (the share of that arm's samples that succeeded on that seed), the paired differences are
tested with an exact sign-flip permutation test (all 2^5 flips), and the mean difference gets a bootstrap interval over seeds.

In [8]:
print("(a) pooled 2x2, demo vs nodemo")
for scene in SCENES:
    k1 = sum(r["ok"] for r in evals if r["arm"] == "demo" and r["scene"] == scene); n1 = sum(1 for r in evals if r["arm"] == "demo" and r["scene"] == scene)
    k2 = sum(r["ok"] for r in evals if r["arm"] == "nodemo" and r["scene"] == scene); n2 = sum(1 for r in evals if r["arm"] == "nodemo" and r["scene"] == scene)
    d, lo, hi = newcombe_diff(k1, n1, k2, n2)
    p = fisher_exact(k1, n1 - k1, k2, n2 - k2)
    print(f"  {scene:7s} demo {k1}/{n1} vs nodemo {k2}/{n2}: diff {100*d:+.0f} pts (Newcombe 95% [{100*lo:+.0f}, {100*hi:+.0f}]), Fisher exact p = {p:.4f}")

print()
print("(b) paired by seed (seed = unit of analysis)")
by = defaultdict(lambda: defaultdict(list))
for r in evals:
    by[(r["arm"], r["scene"])][r["seed"]].append(r["ok"])
rng = random.Random(0)
PAIRED = {}
for scene in SCENES:
    seeds = sorted(by[("demo", scene)])
    sc = {a: {s: sum(by[(a, scene)][s]) / len(by[(a, scene)][s]) for s in seeds} for a in ARMS}
    diffs = [sc["demo"][s] - sc["nodemo"][s] for s in seeds]
    obs = sum(diffs) / len(diffs)
    flips = [sum(sg * d for sg, d in zip(signs, diffs)) / len(diffs)
             for signs in __import__("itertools").product((1, -1), repeat=len(diffs))]
    p_perm = sum(abs(f) >= abs(obs) - 1e-12 for f in flips) / len(flips)
    boots = sorted(sum(rng.choice(diffs) for _ in diffs) / len(diffs) for _ in range(10000))
    lo, hi = boots[249], boots[9749]
    PAIRED[scene] = dict(mean_diff=obs, ci=(lo, hi), p_perm=p_perm, per_seed=dict(zip(seeds, diffs)))
    print(f"  {scene:7s} mean per-seed difference {obs:+.2f} (bootstrap 95% [{lo:+.2f}, {hi:+.2f}]), exact sign-flip p = {p_perm:.4f} "
          f"(minimum attainable with 5 seeds: {2 / 2 ** len(diffs):.4f})")
    print("          per-seed differences:", {s: round(d, 2) for s, d in zip(seeds, diffs)})

(a) pooled 2x2, demo vs nodemo
  scaled  demo 7/10 vs nodemo 8/10: diff -10 pts (Newcombe 95% [-44, +26]), Fisher exact p = 1.0000
  stock   demo 7/10 vs nodemo 0/10: diff +70 pts (Newcombe 95% [+29, +89]), Fisher exact p = 0.0031

(b) paired by seed (seed = unit of analysis)
  scaled  mean per-seed difference -0.10 (bootstrap 95% [-0.40, +0.20]), exact sign-flip p = 1.0000 (minimum attainable with 5 seeds: 0.0625)
          per-seed differences: {1: 0.5, 2: -0.5, 3: -0.5, 4: 0.0, 5: 0.0}
  stock   mean per-seed difference +0.70 (bootstrap 95% [+0.30, +1.00]), exact sign-flip p = 0.1250 (minimum attainable with 5 seeds: 0.0625)
          per-seed differences: {1: 0.5, 2: 1.0, 3: 1.0, 4: 0.0, 5: 1.0}


In [9]:
# Figure 3: mean paired difference
def fig_paired():
    x0, x1, top = 100, 560, 112
    def X(d):
        return x0 + (x1 - x0) * (d + 1) / 2
    b = text(X(-1), top - 10, "← no demo better", 12, INK2) + text(X(1), top - 10, "demo better →", 12, INK2, "end")
    y = top + 24
    for scene in SCENES:
        P = PAIRED[scene]
        lo, hi = P["ci"]
        b += text(x0 - 12, y + 5, scene.capitalize(), 13, INK, "end", 600)
        seen = defaultdict(int)  # seeds with the same value stack, so none hides another
        for d in P["per_seed"].values():
            b += f'<circle cx="{X(d):.1f}" cy="{y + 16 + 10 * seen[d]}" r="4" fill="none" stroke="{INK2}" stroke-width="1.5"/>'
            seen[d] += 1
        b += line(X(lo), y, X(hi), y, INK, 2)
        b += tip(f'<circle cx="{X(P["mean_diff"]):.1f}" cy="{y}" r="8" fill="{INK}" stroke="{SURFACE}" stroke-width="2"/>',
                 f'{scene}: mean difference {P["mean_diff"]:+.2f}, bootstrap 95% {lo:+.2f} to {hi:+.2f}, exact p = {P["p_perm"]:.3f}')
        b += text(x1 + 14, y + 5, f'{P["mean_diff"]:+.2f} [{lo:+.2f}, {hi:+.2f}]', 13, INK)
        y += 76
    for d in (-1, -.5, 0, .5, 1):
        b = line(X(d), top, X(d), y - 18, INK2 if d == 0 else GRID, 1.5 if d == 0 else 1) + b + text(X(d), y, f"{d:+.1f}" if d else "0", 12, INK2, "middle")
    b += text((x0 + x1) / 2, y + 20, "Demo minus no-demo pass share per seed (dot: mean, line: bootstrap 95%, rings: single seeds)", 12, INK2, "middle")
    figure("fig3_paired_by_seed", "Same seeds, compared seed by seed", ["An interval that crosses 0 means no clear difference. Five seeds cannot reach p < 0.05 on the exact test."], b, h=y + 38)
fig_paired()

## 3. The debugging phase (seed 51, scaled scene)

How many attempts each sample needed, and why the failed attempts failed. A sample that never solved the debug seed is still evaluated
with its last program, so the evaluation numbers above include programs that failed on the debug seed.

In [10]:
print(f"{'arm':7s} {'sample':>6s} {'T':>4s} {'attempts':>8s} {'solved':>7s}  attempt reasons")
DEBUG = {}
for a in ARMS:
    for s in sorted({r["sample"] for r in rollouts if r["arm"] == a}):
        rs = sorted((r for r in rollouts if r["arm"] == a and r["sample"] == s), key=lambda r: r["attempt"])
        solved = any(r["success"] for r in rs)
        DEBUG[(a, s)] = dict(attempts=len(rs), solved=solved)
        print(f"{a:7s} {s:>6d} {temps[(a, s)]:>4} {len(rs):>8d} {str(solved):>7s}  " + ", ".join(r["reason"] for r in rs))

arm     sample    T attempts  solved  attempt reasons
demo         0  0.2        2    True  program_completed, task_succeeded
demo         1  0.7        2    True  program_completed, task_succeeded
nodemo       0  0.2        5   False  program_completed, program_error, program_completed, program_completed, program_completed
nodemo       1  0.7        3    True  program_completed, program_error, task_succeeded


In [11]:
# Figure 4: the repair journey
def fig_journey():
    left, step, top = 180, 92, 104
    b = ""
    keys = [(a, s) for a in ARMS for s in sorted({r["sample"] for r in rollouts if r["arm"] == a})]
    for j in range(5):
        b += text(left + j * step, top - 8, "first try" if j == 0 else f"repair {j}", 12, INK2, "middle")
    for i, (a, s) in enumerate(keys):
        y = top + 26 + i * 44
        rs = sorted((r for r in rollouts if r["arm"] == a and r["sample"] == s), key=lambda r: r["attempt"])
        ps = {p["attempt"]: p for p in programs if p["arm"] == a and p["sample"] == s}
        b += f'<circle cx="28" cy="{y}" r="6" fill="{ARM_COLOR[a]}"/>' + text(40, y + 5, f"{ARM_NAME[a]}, T={temps[(a, s)]}", 13, INK)
        b += line(left, y, left + (len(rs) - 1) * step, y, GRID, 2)
        for r in rs:
            cx = left + r["attempt"] * step
            crash = r["reason"] == "program_error"
            fill, ink, sym = (PASS, "#ffffff", "✓") if r["success"] else ("#f6e3e1", FAIL, "!" if crash else "✕")
            tok = ps.get(r["attempt"], {}).get("tokens", 0)
            what = "passed" if r["success"] else ("program crashed" if crash else "task not done")
            b += tip(f'<circle cx="{cx}" cy="{y}" r="14" fill="{fill}" stroke="{SURFACE}" stroke-width="2"/>' + text(cx, y + 5, sym, 14, ink, "middle", 700),
                     f"Attempt {r['attempt']}: {what} ({r['reason']}), {tok:,} tokens")
        solved = any(r["success"] for r in rs)
        b += text(left + len(rs) * step - 60 + 28, y + 5, f"solved, {len(rs)} tries" if solved else f"not solved, {len(rs)} tries", 13, INK, weight=600) if len(rs) < 5 else text(left + 4 * step + 24, y + 5, "not solved" if not solved else "solved", 13, INK, weight=600)
    figure("fig4_repair_journey", "How many tries each program needed on the debug seed", ["✓ passed, ✕ task not done, ! program crashed. Seed 51, scaled scene, at most 4 repairs."], b, h=top + 26 + len(keys) * 44)
fig_journey()

## 4. Cost

Tokens come from the `program` events (one per generated or repaired program); the run total (`run_end.new_tokens`) is the ledger delta for
the whole command and is the number the budget cap applies to. Dollars use the logged `usd_per_mtok`.

In [12]:
print(f"{'arm':7s} {'programs':>8s} {'tokens (sum of program events)':>32s} {'of which reasoning':>20s} {'run new_tokens':>15s} {'USD est.':>9s}")
COST = {}
for a in ARMS:
    ps = [p for p in programs if p["arm"] == a]
    tok = sum(p["tokens"] for p in ps)
    rea = sum(p.get("reasoning_tokens") or 0 for p in ps)
    COST[a] = dict(programs=len(ps), tokens=tok, reasoning=rea, new_tokens=run_end[a]["new_tokens"], usd=run_end[a]["usd_est"])
    print(f"{a:7s} {len(ps):>8d} {tok:>32,d} {rea:>20,d} {run_end[a]['new_tokens']:>15,d} {run_end[a]['usd_est']:>9.2f}")
solved = {a: sum(1 for (aa, _), v in DEBUG.items() if aa == a and v["solved"]) for a in ARMS}
print()
for a in ARMS:
    print(f"{a:7s} solved the debug seed in {solved[a]}/2 samples; new tokens per solved sample: "
          + (f"{COST[a]['new_tokens'] / solved[a]:,.0f}" if solved[a] else "n/a (none solved)"))

arm     programs   tokens (sum of program events)   of which reasoning  run new_tokens  USD est.
demo           4                          204,933              142,617         204,933      2.77
nodemo         8                          247,243               96,795         247,243      3.34

demo    solved the debug seed in 2/2 samples; new tokens per solved sample: 102,466
nodemo  solved the debug seed in 1/2 samples; new tokens per solved sample: 247,243


In [13]:
# Figure 5: tokens per program
def fig_cost():
    left, x1, top, bh = 140, 660, 92, 22
    keys = [(a, s) for a in ARMS for s in sorted({p["sample"] for p in programs if p["arm"] == a})]
    biggest = max(sum(p["tokens"] for p in programs if p["arm"] == a and p["sample"] == s) for a, s in keys)
    def X(v):
        return (x1 - left) * v / biggest
    b = ""
    for i, (a, s) in enumerate(keys):
        y = top + i * 40
        b += text(left - 12, y + 16, f"{ARM_NAME[a]}, T={temps[(a, s)]}", 13, INK, "end")
        x = left
        ps = sorted((p for p in programs if p["arm"] == a and p["sample"] == s), key=lambda p: p["attempt"])
        for p in ps:
            w = max(X(p["tokens"]) - 2, 1)
            b += tip(f'<rect x="{x:.1f}" y="{y}" width="{w:.1f}" height="{bh}" rx="4" fill="{ARM_COLOR[a]}"/>',
                     f'Attempt {p["attempt"]}: {p["tokens"]:,} tokens, of which reasoning {p.get("reasoning_tokens") or 0:,}')
            x += w + 2
        total = sum(p["tokens"] for p in ps)
        b += text(x + 8, y + 16, f"{total / 1000:.0f}k", 13, INK, weight=600)
    usd = {a: run_end[a]["usd_est"] for a in ARMS}
    figure("fig5_tokens", f'Model cost: demo ${usd["demo"]:.2f}, no demo ${usd["nodemo"]:.2f}', ["New tokens per program. Each block is one attempt; hover for details."], b, h=top + len(keys) * 40 + 8)
fig_cost()

## 5. Findings (generated from the numbers above)

In [14]:
def rate(scene, arm):
    k = sum(r["ok"] for r in evals if r["arm"] == arm and r["scene"] == scene)
    return k, sum(1 for r in evals if r["arm"] == arm and r["scene"] == scene)

for scene in SCENES:
    kd, nd = rate(scene, "demo"); kn, nn = rate(scene, "nodemo")
    P = PAIRED[scene]
    print(f"- {scene} scene: demo {kd}/{nd}, no demo {kn}/{nn}; paired by seed {P['mean_diff']:+.2f} "
          f"(bootstrap 95% [{P['ci'][0]:+.2f}, {P['ci'][1]:+.2f}], exact p = {P['p_perm']:.3f}).")
print(f"- Debug seed solved: demo {solved['demo']}/2 samples, no demo {solved['nodemo']}/2 samples.")
print(f"- New tokens: demo {COST['demo']['new_tokens']:,}, no demo {COST['nodemo']['new_tokens']:,}.")
print()
print("Reading rule: with 5 seeds the exact sign-flip test cannot go below p =", f"{2 / 2 ** 5:.4f}", "(two-sided), so it cannot reach p < 0.05 at all.")
print("Judge the paired effect by its bootstrap interval and size, and treat the pooled Fisher p-value above as optimistic (it counts 10 non-independent evaluations).")
print("A claim at the 0.05 level needs more seeds: the held-out range has 50.")

- scaled scene: demo 7/10, no demo 8/10; paired by seed -0.10 (bootstrap 95% [-0.40, +0.20], exact p = 1.000).
- stock scene: demo 7/10, no demo 0/10; paired by seed +0.70 (bootstrap 95% [+0.30, +1.00], exact p = 0.125).
- Debug seed solved: demo 2/2 samples, no demo 1/2 samples.
- New tokens: demo 204,933, no demo 247,243.

Reading rule: with 5 seeds the exact sign-flip test cannot go below p = 0.0625 (two-sided), so it cannot reach p < 0.05 at all.
Judge the paired effect by its bootstrap interval and size, and treat the pooled Fisher p-value above as optimistic (it counts 10 non-independent evaluations).
A claim at the 0.05 level needs more seeds: the held-out range has 50.


## 6. Threats to validity and what would settle them

- **Three differences between the arms, not one.** The demo arm also sees the S4 scene description and E3 evidence. Run a third arm (demo only) to isolate the demonstration.
- **Scene and robot mask are privileged.** `curobo-sim` takes the robot mask from the simulator and `--attach-known` gives the planner the true bowl size. Both arms get the same help, but the absolute numbers are not what a real sensor stack would give.
- **Tiny sample.** Two samples per arm, one run each, five seeds. Intervals are wide. Use seeds 1 to 50 and at least five samples per arm for any claim beyond "signal".
- **Non-independence.** Seeds are shared across samples and arms; use the paired analysis (section 2b) or a mixed model, not the pooled test, for inference.
- **The debug scene is the scaled scene.** The scaled-scene evaluation is therefore a within-distribution test and the stock-scene evaluation is the transfer test.
- **Attach frame bug.** Both arms ran before the fix: the held bowl's points were removed from the cloud, but its spheres sat 9.65 cm too high, inside the wrist, so cuRobo did not model the bowl as an obstacle. Attempts could fail or pass for that reason; a rerun on the fixed code is needed before quoting these numbers as the attached-bowl setting.
- **Planner noise.** cuRobo refusals can be false negatives; a failed rollout may reflect the planner, not the program. Replays of the same program can differ.
- **Acceptance vs held-out.** Debugging used seed 51 (acceptance range 51 to 65). Evaluation used seeds 1 to 5 from the held-out range 1 to 50, which were never used for repair.

## 7. Reproducing

- Settings: `docs/synthesizer/demo-vs-nodemo-config.json`, run book entries `exp12.demo` and `exp12.nodemo`
  (`uv run python -m ants.evalkit.runbook cmd exp12.demo`).
- Raw logs: `data/runs/demo_e3_s4_sim/` and `data/runs/nodemo_task_only_sim/` (git-ignored). The files this notebook reads
  (`events.jsonl`, `config.json`, `summary.md`) are copied next to it in `docs/synthesizer/show-once-vs-task-only/`.

In [15]:
# machine-readable summary for the dashboard / write-up
summary = dict(
    rates={f"{scene}/{a}": dict(k=k, n=n, rate=p, clopper_pearson=cp, wilson=w) for (scene, a, k, n, p, cp, w) in rows},
    paired={s: dict(mean_diff=v["mean_diff"], bootstrap_ci=v["ci"], p_exact=v["p_perm"]) for s, v in PAIRED.items()},
    debug={f"{a}/{s}": v for (a, s), v in DEBUG.items()},
    cost=COST,
)
out = ROOT / "results_summary.json"
out.write_text(json.dumps(summary, indent=2, default=list) + "\n", encoding="utf-8")
print("wrote", out.name)

wrote results_summary.json
